In [ ]:
# =====================================================
# 1. Library
# =====================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import platform
import ast

pd.set_option("display.float_format", "{:.2f}".format)

# OS별 폰트
if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"


# =====================================================
# 2. Data Load
# =====================================================
df = pd.read_csv("../원본 데이터 셋/2025_Airbnb_NYC_listings.csv")

print("데이터 크기:", df.shape)
df.head()


# =====================================================
# 3. Fake Null Check
# =====================================================
fake_null = ['NaN','None','none','NULL','null','N/A','na','']

fake_counts = df.isin(fake_null).sum()
print("Fake null 존재 여부")
print(fake_counts[fake_counts > 0])


# =====================================================
# 4. Datetime 변환
# =====================================================
date_cols = [
    "host_since",
    "first_review",
    "last_review",
    "calendar_last_scraped"
]

df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")


# =====================================================
# 5. Price 타입 변환
# =====================================================
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("$","",regex=False)
    .str.replace(",","",regex=False)
    .astype(float)
)

# 로그 변환
df["price_log"] = np.log1p(df["price"])


# =====================================================
# 6. 리뷰 관련 결측치 처리
# =====================================================
review_zero_cols = [
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "reviews_per_month",
    "estimated_occupancy_l365d"
]

df[review_zero_cols] = df[review_zero_cols].fillna(0)

median_rating = df["review_scores_rating"].median()
df["review_scores_rating"] = df["review_scores_rating"].fillna(median_rating)


# =====================================================
# 7. 숙소 규모 변수 생성
# =====================================================
df["bathrooms"] = df["bathrooms"].fillna(0)
df["bedrooms"] = df["bedrooms"].fillna(0)
df["beds"] = df["beds"].fillna(0)

df["room_capacity"] = (
    df["bathrooms"] +
    df["bedrooms"] +
    df["beds"]
)


# =====================================================
# 8. Amenities 개수 계산
# =====================================================
def get_amenities_len(x):

    try:
        if pd.isna(x):
            return 0
        return len(ast.literal_eval(x))
    except:
        return 0

df["amenities_len"] = df["amenities"].apply(get_amenities_len)


# =====================================================
# 9. 법 규제 분류
# =====================================================
property_map = {

    "Room in hotel": "Lodging",
    "Room in boutique hotel": "Lodging",

    "Entire rental unit": "Residential",
    "Private room in rental unit": "Residential",
    "Private room in home": "Residential",
    "Entire home": "Residential",
    "Entire condo": "Residential",
    "Private room in townhouse": "Residential",
    "Private room in condo": "Residential",
    "Entire townhouse": "Residential",
    "Entire loft": "Residential",
    "Entire guest suite": "Residential"
}

df["property_regulation_type"] = (
    df["property_type"]
    .map(property_map)
    .fillna("Other")
)


# =====================================================
# 10. 합법 여부
# =====================================================
df["legal_flag"] = "Legal"

df.loc[
    (df["property_regulation_type"]=="Residential") &
    (df["room_type"]=="Entire home/apt") &
    (df["minimum_nights"] < 30),
    "legal_flag"
] = "Illegal"


# =====================================================
# 11. 분석 대상 필터링
# =====================================================
df = df[
    (
        (df["property_regulation_type"]=="Residential") &
        (df["legal_flag"]=="Legal")
    )
    |
    (df["property_regulation_type"]=="Lodging")
]


# =====================================================
# 12. 운영 전략 분류
# =====================================================
def label_strategy(row):

    if row["property_regulation_type"]=="Residential":

        if row["minimum_nights"] < 30:
            return "Residential_short_term"

        else:
            return "Residential_long_term"

    else:
        return "Hotel"

df["rental_strategy"] = df.apply(label_strategy, axis=1)


# =====================================================
# 13. 점유율 계산
# =====================================================
df["occupancy_rate"] = (
    df["estimated_occupancy_l365d"] / 365
)


# =====================================================
# 14. 매출 검증
# =====================================================
df["calc_revenue"] = (
    df["price"] *
    df["estimated_occupancy_l365d"]
)

print(
    "매출 계산 일치율:",
    (df["calc_revenue"] == df["estimated_revenue_l365d"]).mean()
)


# =====================================================
# 15. 리뷰 파생 변수
# =====================================================
df["log_reviews"] = np.log1p(df["number_of_reviews"])

df["recent_activity"] = (
    df["number_of_reviews_l30d"] > 0
).astype(int)


# =====================================================
# 16. 운영 기간
# =====================================================
df["operating_days"] = (
    pd.Timestamp("today") -
    df["first_review"]
).dt.days


# =====================================================
# 17. 가격 분위수
# =====================================================
df["price_q"] = pd.qcut(
    df["price"],
    q=4,
    labels=["Q1","Q2","Q3","Q4"]
)


# =====================================================
# 18. 가격 vs 점유율 분석
# =====================================================
sns.scatterplot(
    data=df,
    x="price",
    y="occupancy_rate",
    alpha=0.4
)

plt.title("Price vs Occupancy Rate")
plt.show()